# 18 评分权重与需求扰动敏感性实验

严格执行“物理校核 -> 可行集 -> 评分”的顺序。该实验不允许因偏好权重更高而让物理失效候选重新进入排序。


In [ ]:
from pathlib import Path
from datetime import datetime
import json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
BASE_DIR=Path(r"D:\tabpfn-demo"); OUTPUT_DIR=BASE_DIR/f"18_scoring_sensitivity_{datetime.now().strftime('%Y%m%d_%H%M%S')}"; OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
N_MONTE_CARLO=10_000; DEMAND_SIGMA=.08; SAFETY_SIGMA=.05; rng=np.random.default_rng(42)
SCENARIOS={"balanced":[.35,.35,.20,.10],"lightweight_priority":[.20,.50,.25,.05],"efficiency_priority":[.25,.25,.15,.35],"safety_redundancy_priority":[.55,.25,.10,.10]}
W_NAMES=["safety_margin","mass","volume","efficiency"]
paths=list(BASE_DIR.rglob("03_xgboost_shap_physics_score_*.json"))
if not paths: raise FileNotFoundError("未找到 03_xgboost_shap_physics_score_*.json。")
parents={}
for p in paths: parents.setdefault(p.parent,[]).append(p)
folder,paths=max(parents.items(),key=lambda x:(len(x[1]),max(p.stat().st_mtime for p in x[1])))
packages=[json.loads(p.read_text(encoding="utf-8")) for p in sorted(paths)]
print("证据包目录：",folder,"；工况数：",len(packages))


In [ ]:
def number(x):
    try: return float(x)
    except: return np.nan
def margin(rec,demand,safety):
    comparisons=(rec.get("physics_check") or {}).get("comparisons") or {}
    values=[]
    for v in comparisons.values():
        if isinstance(v,dict):
            cap,req=number(v.get("capability")),number(v.get("requirement"))
            if np.isfinite(cap) and np.isfinite(req) and req>0: values.append(cap/(req*demand*safety))
    return min(values) if values else 0.
def spec(specs,words):
    for k,v in (specs or {}).items():
        if any(x in str(k).lower() for x in words):
            z=number(v)
            if np.isfinite(z): return z
    return np.nan
def benefit(a,invert=False):
    a=np.asarray(a,float); good=np.isfinite(a)
    if not good.any(): return np.full(len(a),.5)
    a=np.where(good,a,np.nanmedian(a[good])); lo,hi=a.min(),a.max()
    z=np.full(len(a),.5) if math.isclose(lo,hi) else (a-lo)/(hi-lo)
    return 1-z if invert else z
def rank(records,demand,safety,w):
    z=[]
    for r in records:
        m=margin(r,demand,safety)
        if m>=1: # 硬约束：不可行候选不评分
            s=r.get("specs") or {}
            z.append({"record":r,"margin":m,"mass":spec(s,["质量","mass"]),"volume":spec(s,["体积","volume"]),"eff":spec(s,["效率","efficiency"])})
    if not z:return []
    d=pd.DataFrame([{k:v for k,v in x.items() if k!="record"} for x in z])
    scores=np.column_stack([np.minimum(d.margin/2,1),benefit(d.mass,True),benefit(d.volume,True),benefit(d.eff)]) @ np.asarray(w)
    for x,v in zip(z,scores):x["score"]=float(v)
    return sorted(z,key=lambda x:x["score"],reverse=True)
def simulate(pkg,scenario,w):
    records=pkg.get("candidate_records") or []; baseline=rank(records,1,1,w)
    baseline_id=baseline[0]["record"].get("candidate_id") if baseline else None; result=[]
    for i in range(N_MONTE_CARLO):
        demand=max(.1,rng.normal(1,DEMAND_SIGMA)); safety=max(.1,rng.normal(1,SAFETY_SIGMA)); weights=rng.dirichlet(np.maximum(np.asarray(w)*80,.5))
        r=rank(records,demand,safety,weights); pick=r[0]["record"] if r else {}
        brank=next((j+1 for j,x in enumerate(r) if x["record"].get("candidate_id")==baseline_id),np.nan)
        result.append({"condition":(pkg.get("joint_requirements") or {}).get("condition","unknown"),"scenario":scenario,"run":i,
          "demand_multiplier":demand,"safety_multiplier":safety,"feasible_count":len(r),"baseline_id":baseline_id,
          "selected_id":pick.get("candidate_id"),"selected_drive_type":pick.get("drive_type"),"baseline_rank":brank,
          **{f"w_{n}":float(x) for n,x in zip(W_NAMES,weights)}})
    return pd.DataFrame(result)


In [ ]:
runs=[]
for pkg in packages:
    for name,w in SCENARIOS.items():
        print((pkg.get("joint_requirements") or {}).get("condition"),name); runs.append(simulate(pkg,name,w))
mc=pd.concat(runs,ignore_index=True); mc["retained"]=mc.selected_id.eq(mc.baseline_id)
summary=mc.groupby(["condition","scenario"]).agg(top1_retention=("retained","mean"),rank_flip_probability=("retained",lambda x:1-x.mean()),
    average_baseline_rank=("baseline_rank","mean"),feasible_count=("feasible_count","mean")).reset_index()
ratios=mc.groupby(["condition","scenario","selected_drive_type"]).size().rename("count").reset_index(); ratios["selection_ratio"]=ratios.groupby(["condition","scenario"])["count"].transform(lambda x:x/x.sum())
flips=mc[~mc.retained].sort_values("run").groupby(["condition","scenario"],as_index=False).first()
mc.to_csv(OUTPUT_DIR/"monte_carlo_runs.csv",index=False,encoding="utf-8-sig"); summary.to_csv(OUTPUT_DIR/"stability_summary.csv",index=False,encoding="utf-8-sig")
ratios.to_csv(OUTPUT_DIR/"drive_type_selection_ratio.csv",index=False,encoding="utf-8-sig"); flips.to_csv(OUTPUT_DIR/"first_rank_flip_cases.csv",index=False,encoding="utf-8-sig")
display(summary); display(ratios.head(20))
p=summary.pivot(index="condition",columns="scenario",values="top1_retention"); fig,ax=plt.subplots(figsize=(9,max(3,.6*len(p))))
im=ax.imshow(p,vmin=0,vmax=1,cmap="YlGn"); ax.set_xticks(range(len(p.columns)),p.columns,rotation=25,ha="right"); ax.set_yticks(range(len(p.index)),p.index)
for i in range(len(p.index)):
 for j in range(len(p.columns)): ax.text(j,i,f"{p.iloc[i,j]:.2f}",ha="center",va="center")
fig.colorbar(im,ax=ax,label="Top-1 保留率"); plt.tight_layout(); plt.savefig(OUTPUT_DIR/"retention_heatmap.png",dpi=220); plt.show()
print("完成。报告保留率、翻转概率、平均名次、可行数及首次翻转的需求/权重条件。",OUTPUT_DIR)
